# Word + Character TF-IDF Class-Weighted SVM

This notebook trains separate Hindi and Bengali stance detection models using combined word and character TF-IDF features and a class-weighted Linear SVM.

**Labels:** 0 = Against, 1 = Favour, 2 = None.

**Dataset requirement:** Place the six cleaned CSV files in `/content/data/`: `hindi_train.csv`, `hindi_val.csv`, `hindi_test.csv`, `bengali_train.csv`, `bengali_val.csv`, `bengali_test.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack

DATA_DIR = "/content/data"
RESULTS_DIR = "/content/results/Word_Character_TFIDF_Class_Weighted_SVM"
os.makedirs(RESULTS_DIR, exist_ok=True)

languages = ["hindi", "bengali"]
labels = [0, 1, 2]
label_names = ["Against", "Favour", "None"]
metrics_list = []


In [ ]:
for language in languages:
    train_df = pd.read_csv(f"{DATA_DIR}/{language}_train.csv")
    val_df = pd.read_csv(f"{DATA_DIR}/{language}_val.csv")
    test_df = pd.read_csv(f"{DATA_DIR}/{language}_test.csv")

    X_train = train_df["Text"].fillna("").astype(str)
    y_train = train_df["Label"]
    X_val = val_df["Text"].fillna("").astype(str)
    y_val = val_df["Label"]
    X_test = test_df["Text"].fillna("").astype(str)
    y_test = test_df["Label"]

    word_vectorizer = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), max_features=50000
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char", ngram_range=(2, 5), max_features=50000
    )

    X_train_word = word_vectorizer.fit_transform(X_train)
    X_train_char = char_vectorizer.fit_transform(X_train)
    X_val_word = word_vectorizer.transform(X_val)
    X_val_char = char_vectorizer.transform(X_val)
    X_test_word = word_vectorizer.transform(X_test)
    X_test_char = char_vectorizer.transform(X_test)

    X_train_combined = hstack([X_train_word, X_train_char])
    X_val_combined = hstack([X_val_word, X_val_char])
    X_test_combined = hstack([X_test_word, X_test_char])

    model = LinearSVC(class_weight="balanced", random_state=42)
    model.fit(X_train_combined, y_train)

    val_predictions = model.predict(X_val_combined)
    test_predictions = model.predict(X_test_combined)

    val_accuracy = accuracy_score(y_val, val_predictions)
    val_macro_f1 = f1_score(y_val, val_predictions, average="macro", zero_division=0)
    test_accuracy = accuracy_score(y_test, test_predictions)
    test_macro_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)
    test_weighted_f1 = f1_score(y_test, test_predictions, average="weighted", zero_division=0)

    metrics_list.append({
        "Language": language.capitalize(),
        "Validation Accuracy": val_accuracy,
        "Validation Macro F1": val_macro_f1,
        "Test Accuracy": test_accuracy,
        "Test Macro F1": test_macro_f1,
        "Test Weighted F1": test_weighted_f1
    })

    print(f"\n{language.upper()} RESULTS")
    print("=" * 45)
    print("Validation Accuracy:", round(val_accuracy, 4))
    print("Validation Macro F1:", round(val_macro_f1, 4))
    print("Test Accuracy:", round(test_accuracy, 4))
    print("Test Macro F1:", round(test_macro_f1, 4))
    print("Test Weighted F1:", round(test_weighted_f1, 4))

    print("\nClassification Report:")
    print(classification_report(
        y_test, test_predictions, labels=labels,
        target_names=label_names, zero_division=0
    ))

    report = classification_report(
        y_test, test_predictions, labels=labels,
        target_names=label_names, output_dict=True, zero_division=0
    )
    pd.DataFrame(report).transpose().to_csv(
        f"{RESULTS_DIR}/{language}_classification_report.csv"
    )

    cm = confusion_matrix(y_test, test_predictions, labels=labels)
    pd.DataFrame(cm, index=label_names, columns=label_names).to_csv(
        f"{RESULTS_DIR}/{language}_confusion_matrix.csv"
    )

    pd.DataFrame({
        "Text": X_test, "Actual": y_test, "Predicted": test_predictions
    }).to_csv(f"{RESULTS_DIR}/{language}_predictions.csv", index=False)

    pd.DataFrame({
        "Text": X_val, "Actual": y_val, "Predicted": val_predictions
    }).to_csv(f"{RESULTS_DIR}/{language}_validation_predictions.csv", index=False)


In [ ]:
metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv(f"{RESULTS_DIR}/metrics_summary.csv", index=False)
print("Final Metrics Summary")
display(metrics_df)
